# Image Classification with Deep Learning
### CIFAR-10 — Exploratory Notebook

This notebook walks through every step of the project interactively:
1. Dataset exploration & visualisation
2. Model building (CNN from scratch)
3. Training with augmentation
4. Evaluation — accuracy, confusion matrix, classification report
5. Transfer learning with MobileNetV2
6. Saving & loading the model

In [ ]:
# ── Setup ────────────────────────────────────────────────────
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.model import CustomCNN, TransferModel
from src.utils import (
    CIFAR10_CLASSES, load_cifar10, get_augmented_generator,
    plot_training_history, plot_confusion_matrix,
    plot_sample_predictions, print_classification_report,
)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 1.  Dataset Exploration

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = load_cifar10(val_split=0.1)

print(f'Train : {x_train.shape}  labels: {y_train.shape}')
print(f'Val   : {x_val.shape}')
print(f'Test  : {x_test.shape}')

In [ ]:
# Show one sample from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for cls_idx, ax in enumerate(axes.flat):
    # pick first image of this class
    sample_idx = np.where(y_train == cls_idx)[0][0]
    ax.imshow(x_train[sample_idx])
    ax.set_title(CIFAR10_CLASSES[cls_idx], fontsize=12)
    ax.axis('off')
plt.suptitle('One sample per class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution (should be perfectly balanced for CIFAR-10)
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar([CIFAR10_CLASSES[i] for i in unique], counts, color='steelblue', edgecolor='white')
plt.title('Training set class distribution', fontsize=13)
plt.xticks(rotation=30, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2.  Data Augmentation Preview

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    horizontal_flip=True, width_shift_range=0.1,
    height_shift_range=0.1, rotation_range=15,
    zoom_range=0.1, channel_shift_range=0.1,
)

sample = x_train[0:1]   # pick one image
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0][0].imshow(sample[0])
axes[0][0].set_title('Original')
axes[0][0].axis('off')

for ax in axes.flat[1:]:
    aug = datagen.random_transform(sample[0])
    ax.imshow(np.clip(aug, 0, 1))
    ax.set_title('Augmented')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3.  Custom CNN — Build & Train

In [ ]:
cnn_model = CustomCNN(input_shape=(32, 32, 3), num_classes=10).build()
cnn_model.summary()

In [ ]:
EPOCHS     = 30   # increase for better results
BATCH_SIZE = 64

train_gen = get_augmented_generator(x_train, y_train, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1),
]

history = cnn_model.fit(
    train_gen,
    steps_per_epoch=len(x_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

## 4.  Evaluation

In [ ]:
plot_training_history(history, save_path='../results/nb_training_curves.png')

# Show inline
from IPython.display import Image as IPImage
IPImage('../results/nb_training_curves.png')

In [ ]:
test_loss, test_acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy : {test_acc*100:.2f}%')
print(f'Test loss     : {test_loss:.4f}')

In [ ]:
y_pred = np.argmax(cnn_model.predict(x_test, verbose=0), axis=1)

plot_confusion_matrix(y_test, y_pred, CIFAR10_CLASSES, save_path='../results/nb_cm.png')
IPImage('../results/nb_cm.png')

In [ ]:
print_classification_report(y_test, y_pred, CIFAR10_CLASSES, save_path='../results/nb_report.txt')

In [ ]:
plot_sample_predictions(x_test, y_test, y_pred, CIFAR10_CLASSES, save_path='../results/nb_samples.png')
IPImage('../results/nb_samples.png')

## 5.  Transfer Learning — MobileNetV2

In [ ]:
# Reload data at 96×96 for MobileNetV2
(x_tr96, y_tr96), (x_v96, y_v96), (x_te96, y_te96) = load_cifar10(
    val_split=0.1, target_size=(96, 96)
)

tl_obj   = TransferModel(input_shape=(96, 96, 3), num_classes=10)
tl_model = tl_obj.build()
tl_model.summary()

In [ ]:
# Phase 1 – warm up the head (backbone frozen)
tl_gen = get_augmented_generator(x_tr96, y_tr96, batch_size=64)

tl_history = tl_model.fit(
    tl_gen,
    steps_per_epoch=len(x_tr96) // 64,
    epochs=15,
    validation_data=(x_v96, y_v96),
    verbose=1,
)

# Phase 2 – unfreeze & fine-tune at low LR
tl_obj.unfreeze(n_layers=30, learning_rate=1e-5)

ft_history = tl_model.fit(
    tl_gen,
    steps_per_epoch=len(x_tr96) // 64,
    epochs=10,
    validation_data=(x_v96, y_v96),
    verbose=1,
)

In [ ]:
tl_loss, tl_acc = tl_model.evaluate(x_te96, y_te96, verbose=0)
print(f'MobileNetV2 Test accuracy : {tl_acc*100:.2f}%')

## 6.  Save & Load Model

In [ ]:
# Save
os.makedirs('../models', exist_ok=True)
cnn_model.save('../models/cnn_notebook.keras')
print('Model saved!')

# Load & verify
loaded = tf.keras.models.load_model('../models/cnn_notebook.keras')
loaded_loss, loaded_acc = loaded.evaluate(x_test, y_test, verbose=0)
print(f'Loaded model accuracy: {loaded_acc*100:.2f}%  ✓')